<a href="https://colab.research.google.com/github/postnicov/ResazurinResorufin/blob/main/Resazurin_Resorufin_Color_Mixing_Reflection_v5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Resazurin / Resorufin pH-dependent color mixing (Reflection, v5)

This Colab notebook retrieves pH-dependent resazurin and resorufin absorbance spectra, mixes them at selected resorufin fractions, converts the mixtures to diffuse reflectance using wavelength-dependent Kubelka-Munk scattering, and exports CIE L*a*b*, sRGB, and Munsell results to Excel.

**v5 update:** Munsell codes are now calculated under **Illuminant C**. The spectrum is first evaluated under D65 to produce XYZ(D65), then XYZ(D65) is chromatically adapted to XYZ(C) using the **Bradford Von Kries** transform. The converted XYZ(C) is passed to the standard Munsell renotation lookup. The `Munsell note` column separately records `nearest valid` when gamut correction is necessary.

In [1]:
#@title Install and import required packages
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    'colour-science': 'colour',
    'pandas': 'pandas',
    'numpy': 'numpy',
    'openpyxl': 'openpyxl',
    'XlsxWriter': 'xlsxwriter',
}

missing_packages = [package for package, module in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    print('Installing:', ', '.join(missing_packages))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *missing_packages])
else:
    print('All required packages are already installed.')

import re
import warnings
import colour
import numpy as np
import pandas as pd
from colour.notation import xyY_to_munsell_colour
from colour.adaptation import chromatic_adaptation_VonKries
from colour.utilities import ColourUsageWarning

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

warnings.filterwarnings('ignore', category=ColourUsageWarning)
print('Libraries imported successfully.')

Installing: colour-science, XlsxWriter
Libraries imported successfully.


In [2]:
#@title Set data source URLs and load the spectra tables
RESAZURIN_URL = 'https://raw.githubusercontent.com/postnicov/ResazurinResorufin/refs/heads/main/Data/pHRzSpectra.csv'  #@param {type:"string"}
RESORUFIN_URL = 'https://raw.githubusercontent.com/postnicov/ResazurinResorufin/refs/heads/main/Data/pHRfSpectra.csv'  #@param {type:"string"}

def load_spectra(url):
    """Load a spectra CSV whose first row gives pH and first column gives wavelength."""
    raw = pd.read_csv(url, header=0, index_col=0)
    raw.columns = [float(column) for column in raw.columns]
    raw.index = raw.index.astype(float)
    return raw.sort_index()

resazurin_spectra = load_spectra(RESAZURIN_URL)
resorufin_spectra = load_spectra(RESORUFIN_URL)
common_pH = sorted(set(resazurin_spectra.columns).intersection(resorufin_spectra.columns))
if not common_pH:
    raise ValueError('No common pH values were found between the two spectra tables.')

print(f'Resazurin spectrum: {resazurin_spectra.shape[0]} wavelengths x {resazurin_spectra.shape[1]} pH values')
print(f'Resorufin spectrum: {resorufin_spectra.shape[0]} wavelengths x {resorufin_spectra.shape[1]} pH values')
print(f'Common pH values ({len(common_pH)}):', common_pH)

Resazurin spectrum: 341 wavelengths x 17 pH values
Resorufin spectrum: 341 wavelengths x 17 pH values
Common pH values (17): [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 7.5, 8.0, 8.5, 9.0]


In [3]:
#@title Set the resorufin fraction range (x)
X_MIN = 0.0  #@param {type:"number"}
X_STEP = 0.2  #@param {type:"number"}
X_MAX = 1.0  #@param {type:"number"}

if X_STEP <= 0:
    raise ValueError('X_STEP must be positive.')
if X_MAX < X_MIN:
    raise ValueError('X_MAX must be greater than or equal to X_MIN.')

n_steps = int(round((X_MAX - X_MIN) / X_STEP))
x_values = [round(X_MIN + i * X_STEP, 10) for i in range(n_steps + 1)]
if x_values[-1] < X_MAX and not np.isclose(x_values[-1], X_MAX):
    x_values.append(round(X_MAX, 10))
x_values = sorted(set(x_values))
if any(x < 0 or x > 1 for x in x_values):
    raise ValueError('All x values must lie between 0 and 1.')

print(f'Resorufin fraction values x ({len(x_values)}):', x_values)

Resorufin fraction values x (6): [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]


In [4]:
#@title Set wavelength-dependent Kubelka-Munk scattering parameters
S0 = 100  #@param {type:"number"}
s = 0.004  #@param {type:"number"}

if S0 <= 0:
    raise ValueError('S0 must be positive.')
if s < 0:
    raise ValueError('s must be non-negative.')

print('Scattering function: S(lambda) = S0 * exp[-s * (lambda - 360)]')
print(f'S0 = {S0}; s = {s}')

Scattering function: S(lambda) = S0 * exp[-s * (lambda - 360)]
S0 = 100; s = 0.004


In [5]:
#@title Set Munsell Illuminant C chromatic-adaptation settings
OBSERVER = 'CIE 1931 2 Degree Standard Observer'
CAT_TRANSFORM = 'Bradford'  #@param ['Bradford', 'CAT02', 'CAT16', 'Von Kries']

xy_C = colour.CCS_ILLUMINANTS[OBSERVER]['C']
xy_D65 = colour.CCS_ILLUMINANTS[OBSERVER]['D65']
XYZ_w_C = colour.xy_to_XYZ(xy_C)
XYZ_w_D65 = colour.xy_to_XYZ(xy_D65)

print('Munsell renotation illuminant: C')
print('Chromatic adaptation: Von Kries with', CAT_TRANSFORM, 'transform')
print('C white point xy:', xy_C)
print('D65 white point xy:', xy_D65)

Munsell renotation illuminant: C
Chromatic adaptation: Von Kries with Bradford transform
C white point xy: [ 0.31006  0.31616]
D65 white point xy: [ 0.3127  0.329 ]


In [6]:
#@title Define spectral mixing, Kubelka-Munk, colorimetry, and Munsell Illuminant C functions
ILLUMINANT = colour.SDS_ILLUMINANTS['D65']
CMFS = colour.MSDS_CMFS[OBSERVER]
D65_XY = xy_D65

def mix_absorbance(abs_rz, abs_rf, x):
    """Linearly mix absorbance: (1 - x) * resazurin + x * resorufin."""
    return (1.0 - x) * abs_rz + x * abs_rf

def scattering_coefficient(wavelengths, S0, s):
    """Return S(lambda) = S0 * exp[-s * (lambda - 360)]."""
    wavelengths = np.asarray(wavelengths, dtype=float)
    return S0 * np.exp(-s * (wavelengths - 360.0))

def absorbance_to_reflectance(absorbance, wavelengths, S0, s):
    """Use K(lambda)/S(lambda) in the optically thick Kubelka-Munk reflectance relation."""
    scattering = scattering_coefficient(wavelengths, S0, s)
    k_over_s = np.clip(np.asarray(absorbance, dtype=float), 0.0, None) / scattering
    reflectance = 1.0 + k_over_s - np.sqrt(k_over_s ** 2 + 2.0 * k_over_s)
    return np.clip(reflectance, 0.0, 1.0)

def spectrum_to_lab_rgb(wavelengths, reflectance):
    """Convert reflectance to CIE Lab, 8-bit sRGB, and XYZ under D65."""
    sd = colour.SpectralDistribution(dict(zip(wavelengths, reflectance)))
    sd = sd.align(colour.SpectralShape(int(wavelengths.min()), int(wavelengths.max()), 1))
    xyz_d65 = colour.sd_to_XYZ(sd, cmfs=CMFS, illuminant=ILLUMINANT) / 100.0
    lab = colour.XYZ_to_Lab(xyz_d65, D65_XY)
    rgb = colour.XYZ_to_sRGB(xyz_d65)
    rgb_8bit = np.clip(np.round(rgb * 255.0), 0, 255).astype(int)
    return lab, rgb_8bit, xyz_d65

HUE_SECTORS = ['R', 'YR', 'Y', 'GY', 'G', 'BG', 'B', 'PB', 'P', 'RP']

def format_munsell_number(number):
    """Format a Munsell number without a trailing decimal zero where integral."""
    number = float(number)
    if np.isclose(number, round(number)):
        return str(int(round(number)))
    return f'{number:.2f}'.rstrip('0').rstrip('.')

def normalise_zero_hue_to_previous_sector(munsell_notation, tolerance=1e-8):
    """Convert a 0 hue in one sector to 10 of the preceding sector (e.g., 0PB to 10B)."""
    text = str(munsell_notation).strip().upper()
    match = re.fullmatch(
        r'([+-]?(?:\d+(?:\.\d*)?|\.\d+))\s*([A-Z]+)\s+([^\s/]+)\s*/\s*([^\s]+)',
        text,
    )
    if match is None:
        raise ValueError(f'Cannot parse Munsell notation: {munsell_notation!r}')
    hue_number_text, sector, value_text, chroma_text = match.groups()
    hue_number = float(hue_number_text)
    if np.isclose(hue_number, 0.0, atol=tolerance) and sector in HUE_SECTORS:
        prior_sector = HUE_SECTORS[(HUE_SECTORS.index(sector) - 1) % len(HUE_SECTORS)]
        return f'10{prior_sector} {value_text}/{chroma_text}'
    return f'{format_munsell_number(hue_number)}{sector} {value_text}/{chroma_text}'

def xyz_d65_to_munsell_c(xyz_d65):
    """Adapt XYZ(D65) to XYZ(C), then return Munsell(C) notation and a correction note.

    This follows the attached reference notebook: Bradford Von Kries adaptation
    from D65 white to C white, XYZ(C) to xyY(C), then Munsell renotation lookup.
    """
    xyz_d65 = np.asarray(xyz_d65, dtype=float)
    xyz_c = chromatic_adaptation_VonKries(
        xyz_d65, XYZ_w_D65, XYZ_w_C, transform=CAT_TRANSFORM
    )
    try:
        raw_munsell = xyY_to_munsell_colour(colour.XYZ_to_xyY(xyz_c))
        return normalise_zero_hue_to_previous_sector(raw_munsell), ''
    except Exception:
        pass

    # For values outside the Munsell renotation lattice, minimally desaturate
    # in D65 Lab, then adapt each candidate to C before lookup.
    lab_d65 = colour.XYZ_to_Lab(xyz_d65, D65_XY)
    clipped_lab = np.array([np.clip(lab_d65[0], 10.0, 90.0), lab_d65[1], lab_d65[2]])
    for chroma_scale in np.linspace(1.0, 0.0, 101):
        candidate_lab_d65 = np.array([clipped_lab[0], clipped_lab[1] * chroma_scale, clipped_lab[2] * chroma_scale])
        candidate_xyz_d65 = colour.Lab_to_XYZ(candidate_lab_d65, D65_XY)
        candidate_xyz_c = chromatic_adaptation_VonKries(
            candidate_xyz_d65, XYZ_w_D65, XYZ_w_C, transform=CAT_TRANSFORM
        )
        try:
            raw_munsell = xyY_to_munsell_colour(colour.XYZ_to_xyY(candidate_xyz_c))
            return normalise_zero_hue_to_previous_sector(raw_munsell), 'nearest valid'
        except Exception:
            continue
    return 'Unavailable', 'nearest valid'

def round_to_half(value):
    """Round a positive decimal to nearest 0.5 using half-up rounding."""
    return np.floor(float(value) * 2.0 + 0.5) / 2.0

def format_half_step(value):
    """Format a half-step without .0 for whole numbers."""
    rounded_value = round_to_half(value)
    if np.isclose(rounded_value, round(rounded_value)):
        return str(int(round(rounded_value)))
    return f'{rounded_value:.1f}'

def munsell_to_half_step(munsell_code):
    """Round a Munsell(C) hue, value, and chroma to the nearest 0.5."""
    neutral_match = re.fullmatch(r'N\s+(\d+(?:\.\d+)?)/', munsell_code)
    if neutral_match:
        return f'N {format_half_step(neutral_match.group(1))}/'
    chromatic_match = re.fullmatch(
        r'(\d+(?:\.\d+)?)([A-Z]+)\s+(\d+(?:\.\d+)?)/(\d+(?:\.\d+)?)', munsell_code
    )
    if not chromatic_match:
        return munsell_code
    hue, hue_letter, value, chroma = chromatic_match.groups()
    return f'{format_half_step(hue)}{hue_letter} {format_half_step(value)}/{format_half_step(chroma)}'

print('Core functions defined.')

Core functions defined.


In [7]:
#@title Compute mixed spectra, colors, and Munsell Illuminant C codes for all pH and x combinations
wavelengths_rz = resazurin_spectra.index.to_numpy(dtype=float)
wavelengths_rf = resorufin_spectra.index.to_numpy(dtype=float)
common_wavelengths = np.intersect1d(wavelengths_rz, wavelengths_rf)
if len(common_wavelengths) == 0:
    raise ValueError('The two spectra tables have no wavelengths in common.')

resazurin_spectra = resazurin_spectra.loc[common_wavelengths]
resorufin_spectra = resorufin_spectra.loc[common_wavelengths]
wavelengths = common_wavelengths.astype(float)
S_lambda = scattering_coefficient(wavelengths, S0, s)

results = []
for pH in common_pH:
    abs_rz = resazurin_spectra[pH].to_numpy(dtype=float)
    abs_rf = resorufin_spectra[pH].to_numpy(dtype=float)
    for x in x_values:
        mixed_abs = mix_absorbance(abs_rz, abs_rf, x)
        reflectance = absorbance_to_reflectance(mixed_abs, wavelengths, S0, s)
        lab, rgb, xyz_d65 = spectrum_to_lab_rgb(wavelengths, reflectance)
        munsell_raw, munsell_note = xyz_d65_to_munsell_c(xyz_d65)
        results.append({
            'pH': pH,
            'x': x,
            'L*': round(float(lab[0]), 3),
            'a*': round(float(lab[1]), 3),
            'b*': round(float(lab[2]), 3),
            'R': int(rgb[0]),
            'G': int(rgb[1]),
            'B': int(rgb[2]),
            'Munsell (C)': munsell_to_half_step(munsell_raw),
            'Munsell note': munsell_note,
        })

results_df = pd.DataFrame(results)
print(f'Computed {len(results_df)} pH × x combinations using {len(wavelengths)} common wavelengths.')
print(f'S(lambda) range across the spectrum: {S_lambda.min():.6f} to {S_lambda.max():.6f}')

reference = results_df[np.isclose(results_df['pH'], 6.5) & np.isclose(results_df['x'], 0.0)]
if not reference.empty:
    print('Check — pH = 6.5, x = 0, Munsell (C):', reference.iloc[0]['Munsell (C)'])
display(results_df.head(10))

Computed 102 pH × x combinations using 341 common wavelengths.
S(lambda) range across the spectrum: 25.666078 to 100.000000
Check — pH = 6.5, x = 0, Munsell (C): 5.5PB 9.5/3


,pH,x,L*,a*,b*,R,G,B,Munsell (C),Munsell note
0,1.0,0.0,97.167,4.807,0.704,255,244,246,8RP 9.5/4,
1,1.0,0.2,97.350,4.336,0.954,255,245,246,9.5RP 9.5/3.5,
2,1.0,0.4,97.551,3.839,1.231,255,245,246,1.5R 10/3.5,
3,1.0,0.6,97.775,3.298,1.542,255,246,246,4R 9/1,nearest valid
4,1.0,0.8,98.031,2.695,1.904,255,247,246,7R 9/1,nearest valid
5,1.0,1.0,98.342,1.996,2.351,255,249,246,1.5YR 9/0.5,nearest valid
6,1.5,0.0,96.839,4.640,-0.004,255,243,246,5.5RP 9.5/3.5,
7,1.5,0.2,97.052,4.118,0.392,255,244,246,7RP 9.5/3.5,
8,1.5,0.4,97.288,3.576,0.824,255,245,246,9.5RP 9.5/3,
9,1.5,0.6,97.554,2.993,1.301,255,246,246,3.5R 10/3,


In [8]:
#@title Build and download the Excel file with colored swatches and Munsell Illuminant C codes
from datetime import datetime

OUTPUT_FILENAME = f"Resazurin_Resorufin_colors_Reflection_v5_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"

export_df = results_df[['pH', 'x', 'L*', 'a*', 'b*']].copy()
export_df['Color'] = ''
export_df['R'] = results_df['R']
export_df['G'] = results_df['G']
export_df['B'] = results_df['B']
export_df['Munsell (C)'] = results_df['Munsell (C)']
export_df['Munsell note'] = results_df['Munsell note']

output_path = '/content/' + OUTPUT_FILENAME if IN_COLAB else OUTPUT_FILENAME

with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
    export_df.to_excel(writer, sheet_name='Colors', index=False)
    workbook = writer.book
    worksheet = writer.sheets['Colors']
    header_format = workbook.add_format({'bold': True, 'align': 'center', 'valign': 'vcenter', 'border': 1})
    for column_index, column_name in enumerate(export_df.columns):
        worksheet.write(0, column_index, column_name, header_format)

    color_column = export_df.columns.get_loc('Color')
    for row_index, (_, row) in enumerate(results_df.iterrows(), start=1):
        hex_color = '#{:02X}{:02X}{:02X}'.format(int(row['R']), int(row['G']), int(row['B']))
        swatch_format = workbook.add_format({'bg_color': hex_color, 'border': 1})
        worksheet.write_blank(row_index, color_column, None, swatch_format)

    worksheet.set_column('A:A', 8)
    worksheet.set_column('B:B', 8)
    worksheet.set_column('C:E', 10)
    worksheet.set_column('F:F', 12)
    worksheet.set_column('G:I', 8)
    worksheet.set_column('J:J', 18)
    worksheet.set_column('K:K', 16)

print(f'Excel file created: {output_path}')
if IN_COLAB:
    files.download(output_path)
display(export_df.head(10))

Excel file created: /content/Resazurin_Resorufin_colors_Reflection_v5_20260818_135933.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,pH,x,L*,a*,b*,Color,R,G,B,Munsell (C),Munsell note
0,1.0,0.0,97.167,4.807,0.704,,255,244,246,8RP 9.5/4,
1,1.0,0.2,97.350,4.336,0.954,,255,245,246,9.5RP 9.5/3.5,
2,1.0,0.4,97.551,3.839,1.231,,255,245,246,1.5R 10/3.5,
3,1.0,0.6,97.775,3.298,1.542,,255,246,246,4R 9/1,nearest valid
4,1.0,0.8,98.031,2.695,1.904,,255,247,246,7R 9/1,nearest valid
5,1.0,1.0,98.342,1.996,2.351,,255,249,246,1.5YR 9/0.5,nearest valid
6,1.5,0.0,96.839,4.640,-0.004,,255,243,246,5.5RP 9.5/3.5,
7,1.5,0.2,97.052,4.118,0.392,,255,244,246,7RP 9.5/3.5,
8,1.5,0.4,97.288,3.576,0.824,,255,245,246,9.5RP 9.5/3,
9,1.5,0.6,97.554,2.993,1.301,,255,246,246,3.5R 10/3,
